# Challenge 1 – Real-Time Weather Alerts Agent

## 1. Environment Setup

This notebook demonstrates an ADK agent that retrieves real-time weather information for locations in the United States.

In [1]:
import getpass
import os

os.environ["GOOGLE_MAPS_API_KEY"] = getpass.getpass(
    "Enter Google Maps API key: "
)

Enter Google Maps API key: ··········


In [2]:
print("Google Maps API key loaded:", bool(os.getenv("GOOGLE_MAPS_API_KEY")))

Google Maps API key loaded: True


## 3. Google Maps Geocoding Tool

This function uses the Google Maps Geocoding API to convert a place name into latitude and longitude coordinates.

In [3]:
import os
import requests


def geocode_location(location: str) -> dict:
    """Convert a location name to latitude and longitude coordinates.

    Args:
        location: A city, state, or other location in the United States.

    Returns:
        A dictionary containing the formatted location, latitude, and longitude.
    """
    api_key = os.environ["GOOGLE_MAPS_API_KEY"]

    url = "https://maps.googleapis.com/maps/api/geocode/json"

    params = {
        "address": location,
        "key": api_key,
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    data = response.json()

    if data["status"] != "OK":
        return {
            "status": "error",
            "message": f"Geocoding failed: {data['status']}",
        }

    result = data["results"][0]
    coordinates = result["geometry"]["location"]

    return {
        "status": "success",
        "location": result["formatted_address"],
        "latitude": coordinates["lat"],
        "longitude": coordinates["lng"],
    }

In [4]:
result = geocode_location("Harrisonburg, VA")
result

{'status': 'success',
 'location': 'Harrisonburg, VA, USA',
 'latitude': 38.4460017,
 'longitude': -78.8697826}

## 4. National Weather Service Tool

This function uses latitude and longitude coordinates to retrieve forecast data from the National Weather Service API at api.weather.gov.

In [5]:
def get_weather(latitude: float, longitude: float) -> dict:
    """Retrieve the weather forecast for a U.S. location.

    Args:
        latitude: Latitude of the location.
        longitude: Longitude of the location.

    Returns:
        A dictionary containing current forecast information from the
        National Weather Service.
    """
    headers = {
        "User-Agent": "ADK Weather Agent Training"
    }

    # Step 1: Ask NWS which forecast endpoint serves these coordinates.
    points_url = (
        f"https://api.weather.gov/points/{latitude},{longitude}"
    )

    points_response = requests.get(
        points_url,
        headers=headers,
        timeout=10,
    )
    points_response.raise_for_status()

    points_data = points_response.json()

    # NWS provides the appropriate forecast URL for this location.
    forecast_url = points_data["properties"]["forecast"]

    # Step 2: Retrieve the actual forecast.
    forecast_response = requests.get(
        forecast_url,
        headers=headers,
        timeout=10,
    )
    forecast_response.raise_for_status()

    forecast_data = forecast_response.json()

    # Grab the first forecast period.
    period = forecast_data["properties"]["periods"][0]

    return {
        "status": "success",
        "period": period["name"],
        "temperature": period["temperature"],
        "temperature_unit": period["temperatureUnit"],
        "wind_speed": period["windSpeed"],
        "wind_direction": period["windDirection"],
        "short_forecast": period["shortForecast"],
        "detailed_forecast": period["detailedForecast"],
    }

In [6]:
location = geocode_location("Harrisonburg, VA")

weather = get_weather(
    location["latitude"],
    location["longitude"],
)

weather

{'status': 'success',
 'period': 'Today',
 'temperature': 82,
 'temperature_unit': 'F',
 'wind_speed': '3 to 7 mph',
 'wind_direction': 'W',
 'short_forecast': 'Sunny',
 'detailed_forecast': 'Sunny, with a high near 82. West wind 3 to 7 mph.'}

## 5. Test the Tools Independently

Test the geocoding and National Weather Service functions for multiple U.S. cities before integrating them with the ADK agent.

In [7]:
test_cities = [
    "Harrisonburg, VA",
    "Denver, CO",
    "Miami, FL",
    "Seattle, WA",
    "New York, NY",
]

for city in test_cities:
    location = geocode_location(city)

    weather = get_weather(
        location["latitude"],
        location["longitude"],
    )

    print(f"\n{location['location']}")
    print(f"Coordinates: {location['latitude']}, {location['longitude']}")
    print(
        f"{weather['period']}: "
        f"{weather['temperature']}°{weather['temperature_unit']} - "
        f"{weather['short_forecast']}"
    )


Harrisonburg, VA, USA
Coordinates: 38.4460017, -78.8697826
Today: 82°F - Sunny

Denver, CO, USA
Coordinates: 39.7392358, -104.990251
Today: 93°F - Mostly Sunny then Chance Showers And Thunderstorms

Miami, FL, USA
Coordinates: 25.7616798, -80.1917902
Today: 89°F - Chance Showers And Thunderstorms then Mostly Sunny

Seattle, WA, USA
Coordinates: 47.6061389, -122.3328481
Today: 76°F - Sunny

New York, NY, USA
Coordinates: 40.7127753, -74.0059728
Today: 82°F - Mostly Sunny then Slight Chance Showers And Thunderstorms


## 6. Create the ADK Weather Agent

The ADK agent will use the geocoding tool to convert a user-supplied location into latitude and longitude, then call the National Weather Service tool to retrieve current forecast information and summarize it for the user.

In [12]:
import os
import google.auth

credentials, project_id = google.auth.default()

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

print("Vertex AI:", os.environ["GOOGLE_GENAI_USE_VERTEXAI"])
print("Project:", os.environ["GOOGLE_CLOUD_PROJECT"])
print("Location:", os.environ["GOOGLE_CLOUD_LOCATION"])

Vertex AI: TRUE
Project: qwiklabs-gcp-02-64fe8ee0c5bc
Location: us-central1


In [13]:
from google.adk.agents import Agent

weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description="Provides weather information and alerts for locations in the United States.",
    instruction="""
    You are a weather assistant for locations in the United States.

    When a user asks about the weather:
    1. Use the geocode_location tool to convert the requested location
       into latitude and longitude.
    2. Use the get_weather tool with those coordinates to retrieve
       weather information from the National Weather Service.
    3. Provide a concise, easy-to-understand weather summary.
    4. Call attention to potentially hazardous or notable weather
       conditions when appropriate.
    5. Do not invent weather information. Base your response on the
       information returned by the tools.
    """,
    tools=[
        geocode_location,
        get_weather,
    ],
)

## 7. Run and Test the ADK Weather Agent

Create an in-memory session and use an ADK Runner to execute the weather agent against user prompts.

In [14]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

session_service = InMemorySessionService()

APP_NAME = "weather_app"
USER_ID = "test_user"
SESSION_ID = "weather_session"

await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

runner = Runner(
    agent=weather_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

In [15]:
async def ask_weather_agent(prompt: str) -> str:
    """Send a prompt to the weather agent and return its final response."""

    content = types.Content(
        role="user",
        parts=[types.Part(text=prompt)],
    )

    final_response = ""

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content,
    ):
        if (
            event.is_final_response()
            and event.content
            and event.content.parts
        ):
            text_parts = [
                part.text
                for part in event.content.parts
                if getattr(part, "text", None)
            ]

            if text_parts:
                final_response = "\n".join(text_parts)

    return final_response

In [16]:
response = await ask_weather_agent(
    "What is the weather in Denver, Colorado?"
)

print(response)

The weather in Denver, Colorado today will be mostly sunny with a high near 93°F, dropping to around 86°F in the afternoon. There is a 40% chance of showers and thunderstorms after 1 PM, with possible rainfall amounts less than a tenth of an inch. Winds will be from the southeast at 3 to 7 mph.


## 8. Multi-City Agent Test

Test the completed ADK weather agent against multiple U.S. cities to demonstrate that it can select and use the geocoding and weather tools to produce location-specific forecasts.

In [17]:
test_cities = [
    "Miami, Florida",
    "Seattle, Washington",
    "Phoenix, Arizona",
    "New York, New York",
]

for city in test_cities:
    prompt = f"What is the weather today in {city}?"

    response = await ask_weather_agent(prompt)

    print("=" * 70)
    print(f"TEST LOCATION: {city}")
    print("=" * 70)
    print(response)
    print()

TEST LOCATION: Miami, Florida
In Miami, Florida today, expect a chance of showers and thunderstorms between 8 AM and noon, then becoming mostly sunny. The high will be near 89°F, but it will feel much hotter with heat index values as high as 105°F. There's a 30% chance of precipitation, with new rainfall amounts between a tenth and a quarter of an inch possible. A southeast wind will blow at 3 to 9 mph.

TEST LOCATION: Seattle, Washington
The weather in Seattle, Washington today will be sunny, with a high near 76°F. A north wind will blow at 6 to 10 mph.

TEST LOCATION: Phoenix, Arizona
In Phoenix, Arizona today, it will be partly sunny with a high near 108°F. Be aware that heat index values could reach as high as 110°F. There will be a light southwest wind at 0 to 5 mph.

TEST LOCATION: New York, New York
In New York, New York today, it will be mostly sunny with a high near 82°F. There is a slight chance of showers and thunderstorms between 2 PM and 5 PM, with a 30% chance of precipit